# DeepWeeds — Phase A: Bước 0-2 (EDA, backbone, ablation)

In [ ]:
import os, sys, platform
REPO_URL = "https://github.com/hdphuc127/K4-Track4-Day2-Deeplearning-Advance.git"
SUB = "submissions/2A202602796-HoDangPhuc/code"
ROOT = "/kaggle/working"
os.chdir(ROOT)
if not os.path.exists("lab"):
    !git clone -q {REPO_URL} lab
else:
    !git -C lab pull -q
CODE = f"{ROOT}/lab/{SUB}"
sys.path.insert(0, CODE)
!pip -q install timm openpyxl
import torch, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, hashlib
os.makedirs("data/labels", exist_ok=True)
# Ảnh: dataset Kaggle ashuraotsuki/deepweeds-fold0 (images.zip nguyên bản Zenodo, MD5 b7b30f96...). Có thể đã được giải nén sẵn.
zips = glob.glob("/kaggle/input/**/images.zip", recursive=True)
if zips:
    h = hashlib.md5()
    with open(zips[0], "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", h.hexdigest()
    os.makedirs("data/images", exist_ok=True)
    !unzip -q -n {zips[0]} -d data/images     # zip phẳng: 17.509 file .jpg ở gốc
    IMAGES_DIR = f"{ROOT}/data/images"
else:
    jp = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
    assert jp, "không thấy ảnh trong /kaggle/input: gắn dataset ashuraotsuki/deepweeds-fold0"
    IMAGES_DIR = os.path.dirname(jp[0])
print("IMAGES_DIR", IMAGES_DIR, len(os.listdir(IMAGES_DIR)))
B_ = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"   # nhãn/fold nguyên bản của tác giả
for n in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{n}.csv {B_}/{n}.csv
!wc -l data/labels/*.csv

In [ ]:
LABELS_DIR = f"{ROOT}/data/labels"
OUT = f"{ROOT}/out"
os.makedirs(OUT, exist_ok=True)
import experiments as E, train as Tr, dataset as D
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
PATHS = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{OUT}/runs",
             pred_dir=f"{OUT}/predictions", curves_dir=f"{OUT}/curves", num_workers=4)
EPOCHS = 12
def cfg(**kw): return Tr.Config(**{**PATHS, "epochs": EPOCHS, **kw})

In [ ]:
tr, va, te = D.load_split(LABELS_DIR)
info = D.check_split(tr, va, te, IMAGES_DIR)
cnt = pd.DataFrame({k: v["Label"].value_counts().sort_index() for k, v in zip(["train", "val", "test"], [tr, va, te])})
cnt.index = D.CLASS_NAMES; cnt["total"] = cnt.sum(1); print(cnt)
os.makedirs(f"{OUT}/eda", exist_ok=True); cnt.to_csv(f"{OUT}/eda/class_counts.csv")
ax = cnt[["train", "val", "test"]].plot.bar(figsize=(9, 3.5), title="Số ảnh theo lớp (fold 0)"); plt.tight_layout()
plt.savefig(f"{OUT}/eda/class_dist.png", dpi=130); plt.close()
print("tỉ lệ lớp lớn/nhỏ:", cnt.total.max() / cnt.total.min())
from PIL import Image
fig, ax = plt.subplots(9, 3, figsize=(7, 20))
for c in range(9):
    for j, f in enumerate(tr[tr.Label == c].Filename.head(3)):
        ax[c, j].imshow(Image.open(f"{IMAGES_DIR}/{f}")); ax[c, j].axis("off")
    ax[c, 0].set_title(D.CLASS_NAMES[c], fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(f"{OUT}/eda/samples.png", dpi=90); plt.close()

In [ ]:
import json, torch
S = E.sanity(cfg(backbone="resnet50"))
json.dump(S, open(f"{OUT}/eda/sanity.json", "w"))
assert abs(S["loss_init"] - S["ln9"]) < 0.8 and S["loss_overfit"] < 0.2, S
ds = D.DeepWeedsDataset(tr.sample(8, random_state=0), IMAGES_DIR, D.build_transforms(True, 224, "color"))
m, s = torch.tensor(D.IMAGENET_MEAN)[:, None, None], torch.tensor(D.IMAGENET_STD)[:, None, None]
fig, ax = plt.subplots(1, 8, figsize=(16, 2.5))
for a, (x, y, _) in zip(ax, ds): a.imshow((x * s + m).clamp(0, 1).permute(1, 2, 0)); a.set_title(D.CLASS_NAMES[y], fontsize=7); a.axis("off")
plt.savefig(f"{OUT}/eda/aug_check.png", dpi=110); plt.close()

In [ ]:
BACKBONES = ["resnet50", "resnext50", "convnext_tiny", "deit_small", "swin_tiny", "efficientnet_b0", "mobilenetv3"]
for i, bb in enumerate(BACKBONES, 1):
    try:
        E.run_resume(cfg(exp_id=f"B{i:02d}_{bb}", backbone=bb, seed=0))
    except Exception as e:   # vd. hết VRAM: ghi lại, chạy tiếp
        print("FAILED", bb, repr(e)); torch.cuda.empty_cache()
        open(f"{OUT}/failed.txt", "a").write(f"{bb}: {e!r}\n")
B = E.collect_runs(PATHS["out_dir"], "B"); B.to_csv(f"{OUT}/backbones.csv", index=False); print(B[["exp_id", "val_macro_f1", "val_top1", "train_s_per_epoch", "params_m", "gmacs"]])

In [ ]:
BEST = "resnet50"   # backbone chuẩn của ablation (chốt từ trước để chạy tự động; lý do ở báo cáo)
T = {
 "T00": ("nền", {}),
 "T01": ("A khởi tạo", dict(init="frozen")),
 "T02": ("A khởi tạo", dict(init="scratch")),
 "T03": ("B augmentation", dict(aug="color")),
 "T04": ("B augmentation", dict(aug="trivial")),
 "T05": ("B augmentation", dict(mix="cutmix")),
 "T06": ("B augmentation", dict(mix="mixup")),
 "T07": ("C loss", dict(loss="ls", label_smoothing=0.1)),
 "T08": ("C loss", dict(loss="focal", focal_gamma=2.0)),
 "T09": ("C loss", dict(loss="ce_weighted", class_weight_beta=0.0)),
 "T10": ("D sampler", dict(sampler="balanced")),
 "T11": ("F EMA", dict(ema_decay=0.995)),
 "T12": ("E LR", dict(lr_head=1e-4)),
}
for k, (ax_, o) in T.items():
    try:
        E.run_resume(cfg(exp_id=k, backbone=BEST, seed=0, **o))
    except Exception as e:
        print("FAILED", k, repr(e)); torch.cuda.empty_cache(); open(f"{OUT}/failed.txt", "a").write(f"{k}: {e!r}\n")
Tdf = E.collect_runs(PATHS["out_dir"], "T")
Tdf["delta_vs_T00"] = Tdf.val_macro_f1 - Tdf.loc[Tdf.exp_id == "T00", "val_macro_f1"].iloc[0]
Tdf["axis"] = Tdf.exp_id.map(lambda k: T[k][0]); Tdf.to_csv(f"{OUT}/training.csv", index=False)
print(Tdf[["exp_id", "axis", "val_macro_f1", "val_top1", "delta_vs_T00", "train_s_per_epoch"]])

In [ ]:
# bỏ checkpoint để output nhẹ (giữ logits, history, predictions, curves)
!find {OUT}/runs -name "best.pt" -delete
!du -sh {OUT}; ls {OUT}